In [ ]:
# ── Cell 1 — Verify GPU ───────────────────────────────
!nvidia-smi

Fri Jun  5 10:33:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              8W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Steps to Get a Clean State and Reload Updated Files

To ensure you're working with the absolute latest versions of your repository files and to clear the notebook's memory, follow these steps:

1.  **Restart the Colab Runtime**: Go to `Runtime > Restart runtime` in the Colab menu. This clears all variables and loaded modules.
2.  **Run the following cells to clean up and re-prepare the environment.**
3.  **Manually re-upload `data_processed.zip`** when prompted by the `files.upload()` cell.
4.  **Run all cells from the beginning** (or `Runtime > Run all`).

#### 1. Clean up existing cloned repository and data

In [ ]:
# Remove the cloned repository directory if it exists
!rm -rf signature-verification

# Remove the data_processed.zip if it exists
!rm -f data_processed.zip

print("Clean up complete. Now proceeding to re-clone and re-setup.")

Clean up complete. Now proceeding to re-clone and re-setup.


#### 2. Re-clone the repository

In [ ]:
!git clone https://github.com/nickfrostcode/signature-verification
%cd signature-verification
print("Repository re-cloned.")

Cloning into 'signature-verification'...
remote: Enumerating objects: 117, done.
remote: Counting objects: 100% (117/117), done.
remote: Compressing objects: 100% (76/76), done.
Receiving objects: 100% (117/117), 46.31 KiB | 5.79 MiB/s, done.
remote: Total 117 (delta 59), reused 92 (delta 34), pack-reused 0 (from 0)
Resolving deltas: 100% (59/59), done.
/content/signature-verification
Repository re-cloned.


#### 3. Re-install dependencies

In [ ]:
!pip install -r requirements.txt -q
!pip install pillow-heif -q
print("Dependencies re-installed.")

Dependencies re-installed.


#### 4. Re-upload and unzip data

**Important**: You will need to manually re-upload `data_processed.zip` when prompted by the `files.upload()` cell below.

In [ ]:
from google.colab import files, drive
import os

print("--- Data Upload/Selection ---")

choice = ''
while choice not in ['U', 'D']:
    print("Do you want to (U)pload 'data_processed.zip' or load it from (D)rive? (U/D):")
    choice = input().upper()

    if choice == 'D':
        print("\nMounting Google Drive...")
        try:
            drive.mount('/content/drive')
            print("Google Drive mounted successfully.")
        except Exception as e:
            print(f"Could not mount Google Drive: {e}. Please choose 'U' to upload.")
            choice = '' # Reset choice to force re-prompt
    elif choice != 'U':
        print("Invalid choice. Please enter 'U' or 'D'.")
        choice = '' # Reset choice to force re-prompt

if choice == 'U':
    print("\nPlease upload 'data_processed.zip' now:")
    uploaded = files.upload()
    if 'data_processed.zip' not in uploaded:
        print("Warning: 'data_processed.zip' was not found among the uploaded files. Ensure you upload the correct file.")
elif choice == 'D':
    print("\nPlease enter the full path to 'data_processed.zip' in your Google Drive.")
    print("Example: /content/drive/MyDrive/my_data/data_processed.zip")
    drive_file_path = input("Enter path: ")

    if not drive_file_path.strip():
        print("No path provided. Please restart the cell and choose 'U' or 'D'.")
    else:
        try:
            if os.path.exists(drive_file_path):
                # Copy the file to the current working directory
                !cp "{drive_file_path}" .
                print(f"Successfully copied '{drive_file_path}' from Google Drive to current directory.")
            else:
                print(f"Error: File not found at '{drive_file_path}'. Please check the path and try again.")
        except Exception as e:
            print(f"An error occurred while copying from Drive: {e}")

Please re-upload 'data_processed.zip' now:


Saving data_processed.zip to data_processed.zip


In [ ]:
!unzip -q data_processed.zip
!echo "Subjects found:"
!ls data_processed/ | wc -l
print("Data re-unzipped. You are now ready to run the rest of your notebook from a clean state.")

Subjects found:
106
Data re-unzipped. You are now ready to run the rest of your notebook from a clean state.


In [ ]:
import sys
sys.path.insert(0, '/content/signature-verification')

from datasets.base_dataset import get_all_subjects, split_subjects
from datasets.pair_dataset import PairDataset
from datasets.single_dataset import SingleDataset

subjects = get_all_subjects()
train_s, val_s, test_s = split_subjects(subjects)
print(f"Train: {len(train_s)} | Val: {len(val_s)} | Test: {len(test_s)}")

pair_ds   = PairDataset(train_s)
single_ds = SingleDataset(train_s)
print(f"Pair dataset:   {len(pair_ds):,} pairs")
print(f"Single dataset: {len(single_ds):,} images")

Train: 84 | Val: 10 | Test: 11
PairDataset built — 7,980 pairs from 84 subjects
  Label 0 (genuine-genuine): 3,780
  Label 1 (genuine-forged):  4,200
SingleDataset built — 1,260 images from 84 subjects
  Label 0 (genuine): 840
  Label 1 (forged):  420
Pair dataset:   7,980 pairs
Single dataset: 1,260 images


In [ ]:
%run training/train_baseline.py

TRAINING — Baseline CNN
Device: cuda

SingleDataset built — 1,260 images from 84 subjects
  Label 0 (genuine): 840
  Label 1 (forged):  420
SingleDataset built — 150 images from 10 subjects
  Label 0 (genuine): 100
  Label 1 (forged):  50

Train batches: 79 | Val batches: 10

  SingleDataset label counts: Counter({0: 840, 1: 420}) | pos_weight=2.0000
Epoch 01/50 | Train Loss: 0.9304 | Train Acc: 0.5103 | Val Loss: 0.9380 | Val Acc: 0.5933
           ✅ Best model saved (val_loss: 0.9380)
Epoch 02/50 | Train Loss: 0.9209 | Train Acc: 0.5294 | Val Loss: 0.9434 | Val Acc: 0.6800
           ⏳ No improvement (1/10)
Epoch 03/50 | Train Loss: 0.9199 | Train Acc: 0.5643 | Val Loss: 0.9420 | Val Acc: 0.6867
           ⏳ No improvement (2/10)
Epoch 04/50 | Train Loss: 0.9068 | Train Acc: 0.5857 | Val Loss: 0.9324 | Val Acc: 0.7000
           ✅ Best model saved (val_loss: 0.9324)
Epoch 05/50 | Train Loss: 0.9115 | Train Acc: 0.5825 | Val Loss: 0.9236 | Val Acc: 0.7133
           ✅ Best model saved

In [ ]:
%run training/train_siamese.py

TRAINING — Siamese Network
Device: cuda

PairDataset built — 7,980 pairs from 84 subjects
  Label 0 (genuine-genuine): 3,780
  Label 1 (genuine-forged):  4,200
PairDataset built — 950 pairs from 10 subjects
  Label 0 (genuine-genuine): 450
  Label 1 (genuine-forged):  500

Train pairs: 7,980 | Val pairs: 950
Train batches: 499 | Val batches: 60

  PairDataset label counts: Counter({1: 4200, 0: 3780}) | pos_weight=0.9000
Epoch 01/50 | Train Loss: 0.6568 | Train Acc: 0.4985 | Val Loss: 0.6560 | Val Acc: 0.4937
           ✅ Best model saved (val_loss: 0.6560)
Epoch 02/50 | Train Loss: 0.6564 | Train Acc: 0.5142 | Val Loss: 0.6553 | Val Acc: 0.4926
           ✅ Best model saved (val_loss: 0.6553)
Epoch 03/50 | Train Loss: 0.6553 | Train Acc: 0.5382 | Val Loss: 0.6526 | Val Acc: 0.5189
           ✅ Best model saved (val_loss: 0.6526)
Epoch 04/50 | Train Loss: 0.6527 | Train Acc: 0.5430 | Val Loss: 0.6360 | Val Acc: 0.6295
           ✅ Best model saved (val_loss: 0.6360)
Epoch 05/50 | Train 

In [ ]:
from google.colab import files
files.download('saved_models/baseline_cnn.pth')
files.download('saved_models/siamese.pth')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>